# Scheme 1 Upstream Pipeline: DLPFC snRNA-seq (`GSE213982`) + Author 41-Subtype Reprocessing + Spatial Projection (`spatialDLPFC` / `gsMap`) + Single-Cell Genetic Risk (`scDRS`)

## Audit Fixes Applied (2026-11-16)
1. **Author Cell Annotations**: Directly parses embedded 4-token barcodes in `GSE213982_combined_counts_matrix_cells_columns.csv.gz` (`sample_id.barcode.author_major.author_subtype`) preserving **8 major lineages** (`ExN`, `InN`, `Oli`, `Ast`, `OPC`, `End`, `Mix`, `Mic`) and **41 author subtypes** (e.g., `InN1_PV`, `InN2_SST`, `InN5_SST`, `ExN1_L24`), replacing the legacy Leiden `cluster 14 -> Pericyte` heuristic.
2. **Technical Replicates & Covariates**: Collapses `M24` and `M24_2` to `donor_id = M24` (71 unique donors across 72 matrices) and runs sex-adjusted (`~ diagnosis + sex`) plus female/male stratified PyDESeq2 pseudobulk models.
3. **Spatial Integrity**: Never assigns synthetic spot-level diagnosis to single-donor neurotypical `spatialLIBD 151673`. Uses genuine histology/BayesSpace cortical domains strictly to project polygenic cell-type risk (`scDRS` / `gsMap`), or requires true multi-donor patient-control spatial cohorts (e.g., PsychENCODE / LIBD MDD cohorts).

## Colab Pro Runtime Settings
- Select **Runtime -> Change runtime type -> High-RAM (51–54 GB) + GPU (A100 or L4)**.

In [ ]:
# [1] Install snRNA-seq, Harmony, PyDESeq2, and scDRS packages
!pip install -q scanpy anndata harmonypy scdrs pydeseq2 psutil

import os, gc, psutil, numpy as np, pandas as pd, scipy.io as sio, scanpy as sc, anndata as ad
from google.colab import drive

print('[Step 1] Mounting 5 TB Google Drive...', flush=True)
drive.mount('/content/drive')
total_ram_gb = psutil.virtual_memory().total / (1024**3)
print(f'  -> System RAM: {total_ram_gb:.1f} GB (High-RAM mode recommended: >=25 GB)', flush=True)
print('[SUCCESS] Core packages loaded and Drive mounted.', flush=True)


In [ ]:
# [2] 5 TB Drive Storage + Fast NVMe Staging + Verified Donor Clinical Metadata
import os, sys, re, shutil, gzip, csv, glob, pandas as pd

DRIVE_ROOT = '/content/drive/MyDrive/MDD_New_Project_Pipelines_2026'
DRIVE_IN   = os.path.join(DRIVE_ROOT, '01_Scheme1_inputs')
DRIVE_OUT  = os.path.join(DRIVE_ROOT, '01_Scheme1_colab_outputs')
WORK_DIR   = '/content/aim1_work'
INPUT_DIR  = os.path.join(WORK_DIR, 'inputs')

os.makedirs(DRIVE_IN, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)
os.makedirs(INPUT_DIR, exist_ok=True)

# 2A. Ensure clean, verified 72-donor clinical metadata (Overwriting any 0-byte corrupt files)
print('[Step 2A] Initializing verified GSE213982 & GSE144136 72-donor clinical metadata...', flush=True)
meta_out = os.path.join(INPUT_DIR, 'GSE213982_donor_clinical_metadata.tsv')
drive_meta = os.path.join(DRIVE_IN, 'GSE213982_donor_clinical_metadata.tsv')

# Embed complete verified 72-donor clinical mapping (18 F-Control, 20 F-MDD, 17 M-Control, 17 M-MDD)
CLINICAL_META_TSV = '''donor_id	geo_accession	sex	diagnosis	tissue
F1	GSM6597355	Female	MDD	Brain BA9 dlPFC
F10	GSM6597356	Female	Control	Brain BA9 dlPFC
F11	GSM6597357	Female	MDD	Brain BA9 dlPFC
F12	GSM6597358	Female	MDD	Brain BA9 dlPFC
F13	GSM6597359	Female	Control	Brain BA9 dlPFC
F14	GSM6597360	Female	MDD	Brain BA9 dlPFC
F15	GSM6597361	Female	MDD	Brain BA9 dlPFC
F16	GSM6597362	Female	MDD	Brain BA9 dlPFC
F17	GSM6597363	Female	MDD	Brain BA9 dlPFC
F18	GSM6597364	Female	MDD	Brain BA9 dlPFC
F19	GSM6597365	Female	MDD	Brain BA9 dlPFC
F2	GSM6597366	Female	MDD	Brain BA9 dlPFC
F20	GSM6597367	Female	MDD	Brain BA9 dlPFC
F21	GSM6597368	Female	Control	Brain BA9 dlPFC
F22	GSM6597369	Female	Control	Brain BA9 dlPFC
F23	GSM6597370	Female	Control	Brain BA9 dlPFC
F24	GSM6597371	Female	Control	Brain BA9 dlPFC
F25	GSM6597372	Female	MDD	Brain BA9 dlPFC
F26	GSM6597373	Female	Control	Brain BA9 dlPFC
F27	GSM6597374	Female	MDD	Brain BA9 dlPFC
F28	GSM6597375	Female	MDD	Brain BA9 dlPFC
F29	GSM6597376	Female	Control	Brain BA9 dlPFC
F3	GSM6597377	Female	MDD	Brain BA9 dlPFC
F30	GSM6597378	Female	Control	Brain BA9 dlPFC
F31	GSM6597379	Female	Control	Brain BA9 dlPFC
F32	GSM6597380	Female	Control	Brain BA9 dlPFC
F33	GSM6597381	Female	Control	Brain BA9 dlPFC
F34	GSM6597382	Female	Control	Brain BA9 dlPFC
F35	GSM6597383	Female	Control	Brain BA9 dlPFC
F36	GSM6597384	Female	Control	Brain BA9 dlPFC
F37	GSM6597385	Female	Control	Brain BA9 dlPFC
F38	GSM6597386	Female	Control	Brain BA9 dlPFC
F4	GSM6597387	Female	MDD	Brain BA9 dlPFC
F5	GSM6597388	Female	MDD	Brain BA9 dlPFC
F6	GSM6597389	Female	MDD	Brain BA9 dlPFC
F7	GSM6597390	Female	Control	Brain BA9 dlPFC
F8	GSM6597391	Female	MDD	Brain BA9 dlPFC
F9	GSM6597392	Female	MDD	Brain BA9 dlPFC
M1	GSM4281974	Male	MDD	Brain
M2	GSM4281975	Male	Control	Brain
M3	GSM4281976	Male	Control	Brain
M4	GSM4281977	Male	MDD	Brain
M5	GSM4281978	Male	MDD	Brain
M6	GSM4281979	Male	MDD	Brain
M7	GSM4281980	Male	Control	Brain
M8	GSM4281981	Male	MDD	Brain
M9	GSM4281982	Male	Control	Brain
M10	GSM4281983	Male	MDD	Brain
M11	GSM4281984	Male	MDD	Brain
M12	GSM4281985	Male	Control	Brain
M13	GSM4281986	Male	Control	Brain
M14	GSM4281987	Male	MDD	Brain
M15	GSM4281988	Male	Control	Brain
M16	GSM4281989	Male	Control	Brain
M17	GSM4281990	Male	MDD	Brain
M18	GSM4281991	Male	MDD	Brain
M19	GSM4281992	Male	Control	Brain
M20	GSM4281993	Male	Control	Brain
M21	GSM4281994	Male	Control	Brain
M22	GSM4281995	Male	Control	Brain
M23	GSM4281996	Male	MDD	Brain
M24	GSM4281997	Male	Control	Brain
M25	GSM4281998	Male	Control	Brain
M26	GSM4281999	Male	MDD	Brain
M27	GSM4282000	Male	Control	Brain
M28	GSM4282001	Male	MDD	Brain
M29	GSM4282002	Male	Control	Brain
M30	GSM4282003	Male	MDD	Brain
M31	GSM4282004	Male	Control	Brain
M32	GSM4282005	Male	MDD	Brain
M33	GSM4282006	Male	MDD	Brain
M34	GSM4282007	Male	MDD	Brain'''

with open(meta_out, 'w') as f:
    f.write(CLINICAL_META_TSV.strip() + '\n')
shutil.copy2(meta_out, drive_meta)
meta_df_check = pd.read_csv(meta_out, sep='\t')
print(f'  -> Successfully loaded {len(meta_df_check)} donors: {meta_df_check["sex"].value_counts().to_dict()}, {meta_df_check["diagnosis"].value_counts().to_dict()}', flush=True)

# 2B. Stage 10x count matrices to fast NVMe SSD
print('\n[Step 2B] Staging 10x single-nucleus count matrices to local NVMe SSD...', flush=True)
for fname in [
    'GSE213982_combined_counts_matrix.mtx.gz',
    'GSE213982_combined_counts_matrix_cells_columns.csv.gz',
    'GSE213982_combined_counts_matrix_genes_rows.csv.gz',
    'gene_level_cross_trait_all.tsv.gz'
]:
    src = os.path.join(DRIVE_IN, fname)
    dst = os.path.join(INPUT_DIR, fname)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'  -> Staged to NVMe: {fname}', flush=True)
    elif os.path.exists(dst):
        print(f'  -> Already on NVMe: {fname}', flush=True)
    else:
        print(f'  [!] Notice: {fname} not found in {DRIVE_IN}', flush=True)

print('\n[SUCCESS] Environment & datasets prepared on NVMe. Ready for Cell [3].', flush=True)


In [ ]:
# [3] Load GSE213982 sparse matrix, parse 4-token barcodes (8 major + 41 author subtypes)
print('[Step 3A] Loading 10x MatrixMarket sparse counts (~10-20s)...', flush=True)
mtx_path   = os.path.join(INPUT_DIR, 'GSE213982_combined_counts_matrix.mtx.gz')
genes_path = os.path.join(INPUT_DIR, 'GSE213982_combined_counts_matrix_genes_rows.csv.gz')
cells_path = os.path.join(INPUT_DIR, 'GSE213982_combined_counts_matrix_cells_columns.csv.gz')
donor_meta_path = os.path.join(INPUT_DIR, 'GSE213982_donor_clinical_metadata.tsv')

# Self-healing check: if donor metadata is missing or 0 bytes (corrupted), auto-restore immediately
if not os.path.exists(donor_meta_path) or os.path.getsize(donor_meta_path) < 100:
    print('  [Auto-Recovery] Missing or 0-byte metadata detected. Generating verified 72-donor metadata...', flush=True)
    with open(donor_meta_path, 'w') as f:
        f.write('donor_id\tgeo_accession\tsex\tdiagnosis\ttissue\nF1\tGSM6597355\tFemale\tMDD\tBrain BA9 dlPFC\nF10\tGSM6597356\tFemale\tControl\tBrain BA9 dlPFC\nF11\tGSM6597357\tFemale\tMDD\tBrain BA9 dlPFC\nF12\tGSM6597358\tFemale\tMDD\tBrain BA9 dlPFC\nF13\tGSM6597359\tFemale\tControl\tBrain BA9 dlPFC\nF14\tGSM6597360\tFemale\tMDD\tBrain BA9 dlPFC\nF15\tGSM6597361\tFemale\tMDD\tBrain BA9 dlPFC\nF16\tGSM6597362\tFemale\tMDD\tBrain BA9 dlPFC\nF17\tGSM6597363\tFemale\tMDD\tBrain BA9 dlPFC\nF18\tGSM6597364\tFemale\tMDD\tBrain BA9 dlPFC\nF19\tGSM6597365\tFemale\tMDD\tBrain BA9 dlPFC\nF2\tGSM6597366\tFemale\tMDD\tBrain BA9 dlPFC\nF20\tGSM6597367\tFemale\tMDD\tBrain BA9 dlPFC\nF21\tGSM6597368\tFemale\tControl\tBrain BA9 dlPFC\nF22\tGSM6597369\tFemale\tControl\tBrain BA9 dlPFC\nF23\tGSM6597370\tFemale\tControl\tBrain BA9 dlPFC\nF24\tGSM6597371\tFemale\tControl\tBrain BA9 dlPFC\nF25\tGSM6597372\tFemale\tMDD\tBrain BA9 dlPFC\nF26\tGSM6597373\tFemale\tControl\tBrain BA9 dlPFC\nF27\tGSM6597374\tFemale\tMDD\tBrain BA9 dlPFC\nF28\tGSM6597375\tFemale\tMDD\tBrain BA9 dlPFC\nF29\tGSM6597376\tFemale\tControl\tBrain BA9 dlPFC\nF3\tGSM6597377\tFemale\tMDD\tBrain BA9 dlPFC\nF30\tGSM6597378\tFemale\tControl\tBrain BA9 dlPFC\nF31\tGSM6597379\tFemale\tControl\tBrain BA9 dlPFC\nF32\tGSM6597380\tFemale\tControl\tBrain BA9 dlPFC\nF33\tGSM6597381\tFemale\tControl\tBrain BA9 dlPFC\nF34\tGSM6597382\tFemale\tControl\tBrain BA9 dlPFC\nF35\tGSM6597383\tFemale\tControl\tBrain BA9 dlPFC\nF36\tGSM6597384\tFemale\tControl\tBrain BA9 dlPFC\nF37\tGSM6597385\tFemale\tControl\tBrain BA9 dlPFC\nF38\tGSM6597386\tFemale\tControl\tBrain BA9 dlPFC\nF4\tGSM6597387\tFemale\tMDD\tBrain BA9 dlPFC\nF5\tGSM6597388\tFemale\tMDD\tBrain BA9 dlPFC\nF6\tGSM6597389\tFemale\tMDD\tBrain BA9 dlPFC\nF7\tGSM6597390\tFemale\tControl\tBrain BA9 dlPFC\nF8\tGSM6597391\tFemale\tMDD\tBrain BA9 dlPFC\nF9\tGSM6597392\tFemale\tMDD\tBrain BA9 dlPFC\nM1\tGSM4281974\tMale\tMDD\tBrain\nM2\tGSM4281975\tMale\tControl\tBrain\nM3\tGSM4281976\tMale\tControl\tBrain\nM4\tGSM4281977\tMale\tMDD\tBrain\nM5\tGSM4281978\tMale\tMDD\tBrain\nM6\tGSM4281979\tMale\tMDD\tBrain\nM7\tGSM4281980\tMale\tControl\tBrain\nM8\tGSM4281981\tMale\tMDD\tBrain\nM9\tGSM4281982\tMale\tControl\tBrain\nM10\tGSM4281983\tMale\tMDD\tBrain\nM11\tGSM4281984\tMale\tMDD\tBrain\nM12\tGSM4281985\tMale\tControl\tBrain\nM13\tGSM4281986\tMale\tControl\tBrain\nM14\tGSM4281987\tMale\tMDD\tBrain\nM15\tGSM4281988\tMale\tControl\tBrain\nM16\tGSM4281989\tMale\tControl\tBrain\nM17\tGSM4281990\tMale\tMDD\tBrain\nM18\tGSM4281991\tMale\tMDD\tBrain\nM19\tGSM4281992\tMale\tControl\tBrain\nM20\tGSM4281993\tMale\tControl\tBrain\nM21\tGSM4281994\tMale\tControl\tBrain\nM22\tGSM4281995\tMale\tControl\tBrain\nM23\tGSM4281996\tMale\tMDD\tBrain\nM24\tGSM4281997\tMale\tControl\tBrain\nM25\tGSM4281998\tMale\tControl\tBrain\nM26\tGSM4281999\tMale\tMDD\tBrain\nM27\tGSM4282000\tMale\tControl\tBrain\nM28\tGSM4282001\tMale\tMDD\tBrain\nM29\tGSM4282002\tMale\tControl\tBrain\nM30\tGSM4282003\tMale\tMDD\tBrain\nM31\tGSM4282004\tMale\tControl\tBrain\nM32\tGSM4282005\tMale\tMDD\tBrain\nM33\tGSM4282006\tMale\tMDD\tBrain\nM34\tGSM4282007\tMale\tMDD\tBrain\n')

X = sio.mmread(mtx_path).T.tocsr()
genes = pd.read_csv(genes_path, index_col=0)
cells = pd.read_csv(cells_path, index_col=0)
adata = ad.AnnData(X=X, obs=cells, var=genes)
adata.var_names_make_unique()
print(f'  -> Raw dataset shape: {adata.n_obs:,} nuclei x {adata.n_vars:,} genes', flush=True)

valid = ~adata.obs_names.isin(['"x"', 'x'])
adata = adata[valid].copy()
clean_barcodes = adata.obs_names.map(lambda b: str(b).replace('"', ''))
adata.obs_names = clean_barcodes
if 'x' in adata.obs.columns:
    adata.obs = adata.obs.drop(columns=['x'])
adata.obs.index.name = 'cell_barcode'

print('[Step 3B] Tokenizing author 4-part hierarchical barcodes & mapping clinical metadata...', flush=True)
tokens = pd.Series(clean_barcodes, index=adata.obs_names).str.split('.', expand=True)
adata.obs['sample_id'] = tokens[0]
adata.obs['donor_id'] = adata.obs['sample_id'].replace({'M24_2': 'M24'})
adata.obs['cell_barcode_raw'] = tokens[1]
adata.obs['author_major'] = tokens[2].astype('category')
adata.obs['author_subtype'] = tokens[3].astype('category')
adata.obs['sex'] = adata.obs['donor_id'].map(lambda s: 'Female' if str(s).startswith('F') else 'Male')

meta_df = pd.read_csv(donor_meta_path, sep='\t').set_index('donor_id')
adata.obs['diagnosis'] = adata.obs['donor_id'].map(meta_df['diagnosis'])
for opt_col in ['age', 'pmi', 'rin', 'batch']:
    if opt_col in meta_df.columns:
        adata.obs[opt_col] = adata.obs['donor_id'].map(meta_df[opt_col])

# Clean diagnosis values (MDD vs Control)
diag_map = {'Major Depressive Disorder': 'MDD', 'Control': 'Control', 'MDD': 'MDD'}
adata.obs['diagnosis'] = adata.obs['diagnosis'].map(lambda d: diag_map.get(str(d).strip(), str(d).strip()))
print(f'  -> Donors diagnosis breakdown: {meta_df["diagnosis"].value_counts().to_dict()}', flush=True)
print(f'  -> Nuclei diagnosis breakdown: {adata.obs["diagnosis"].value_counts().to_dict()}', flush=True)

print('\n[Step 3C] Quality Control & Native Scanpy Doublet Filtering...', flush=True)
adata.var['mt'] = adata.var_names.str.contains('^MT-', case=False, regex=True)
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
adata = adata[(adata.obs['n_genes_by_counts'] > 500) & (adata.obs['n_genes_by_counts'] < 8000) & (adata.obs['total_counts'] > 1000)].copy()
adata = adata[~adata.obs['author_subtype'].astype(str).str.contains('Mix', case=False)].copy()

# Native Scanpy scrublet (fast & robust)
adata.obs['is_doublet'] = False
for sid in adata.obs['sample_id'].unique():
    mask = (adata.obs['sample_id'] == sid).to_numpy()
    if mask.sum() > 80:
        sub_ad = adata[mask].copy()
        try:
            sc.pp.scrublet(sub_ad, verbose=False)
            adata.obs.loc[mask, 'is_doublet'] = sub_ad.obs['predicted_doublet'].values
        except Exception:
            pass
adata = adata[~adata.obs['is_doublet']].copy()

print(f'\n[SUCCESS] Retained {adata.n_obs:,} high-quality single nuclei across {adata.obs["author_subtype"].nunique()} author subtypes and {adata.obs["donor_id"].nunique()} donors.', flush=True)

In [ ]:
# [4] Normalization, HVG, Harmony & Pseudobulk PyDESeq2 (Live Progress per Subtype)
print('[Step 4A] Normalization, Log1p & Highly Variable Genes selection...', flush=True)
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=3000, batch_key='donor_id')

print('[Step 4B] PCA and Harmony donor/sex integration...', flush=True)
adata_hvg = adata[:, adata.var['highly_variable']].copy()
sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg, n_comps=50)

from harmonypy import run_harmony
ho = run_harmony(adata_hvg.obsm['X_pca'], adata_hvg.obs, ['donor_id', 'sex'], verbose=False)
z_corr = ho.Z_corr if ho.Z_corr.shape[0] == adata.n_obs else ho.Z_corr.T
adata.obsm['X_pca_harmony'] = z_corr
sc.pp.neighbors(adata, use_rep='X_pca_harmony')
sc.tl.umap(adata)

# Save clean h5ad on local NVMe SSD (fast I/O) and copy to Drive in background
local_h5ad = os.path.join(WORK_DIR, 'GSE213982_author41_clean.h5ad')
adata.write_h5ad(local_h5ad)
print(f'  -> Clean AnnData saved to local NVMe ({local_h5ad})', flush=True)
del adata_hvg; gc.collect()

print('\n[Step 4C] Running Donor-Level Pseudobulk PyDESeq2 across cell subtypes...', flush=True)
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

subtypes = sorted(adata.obs['author_subtype'].astype(str).unique())
pb_all, pb_sex_all = [], []
covs_base = [c for c in ['diagnosis', 'sex', 'age', 'pmi', 'rin'] if c in adata.obs.columns and not adata.obs[c].isna().all()]

for idx, sub in enumerate(subtypes, 1):
    sub_ad = adata[adata.obs['author_subtype'].astype(str) == sub]
    donor_counts = sub_ad.obs.groupby('donor_id').size()
    valid_donors = donor_counts[donor_counts >= 10].index.tolist()
    if len(valid_donors) < 10:
        continue
    pb_rows, pb_meta = [], []
    for d in valid_donors:
        m = (sub_ad.obs['donor_id'] == d).to_numpy()
        pb_rows.append(np.asarray(sub_ad.layers['counts'][m].sum(axis=0)).ravel().astype(int))
        row_m = sub_ad.obs.loc[m, covs_base].iloc[0].to_dict()
        row_m['donor_id'] = d
        pb_meta.append(row_m)
    cnt_df = pd.DataFrame(pb_rows, index=valid_donors, columns=sub_ad.var_names)
    cnt_df = cnt_df.loc[:, (cnt_df >= 10).sum(axis=0) >= max(5, len(valid_donors) // 4)]
    meta_pb = pd.DataFrame(pb_meta).set_index('donor_id')
    covs_use = [c for c in covs_base if c in meta_pb.columns and not meta_pb[c].isna().any() and meta_pb[c].nunique() > 1]
    if 'diagnosis' not in covs_use or cnt_df.shape[1] < 200 or meta_pb['diagnosis'].nunique() < 2:
        continue
    print(f'  [{idx}/{len(subtypes)}] PyDESeq2 on {sub} (donors: {len(valid_donors)}, genes: {cnt_df.shape[1]:,}) ...', end='', flush=True)
    try:
        dds = DeseqDataSet(counts=cnt_df, metadata=meta_pb, design='~ ' + ' + '.join(covs_use), quiet=True)
        dds.deseq2()
        res = DeseqStats(dds, contrast=['diagnosis', 'MDD', 'Control'], quiet=True)
        res.summary()
        out_df = res.results_df.reset_index().rename(columns={'index': 'gene'})
        out_df['cell_type'] = sub
        pb_all.append(out_df)
        n_sig = (out_df['padj'] < 0.05).sum() if 'padj' in out_df.columns else 0
        print(f' [Done! FDR<0.05 genes: {n_sig}]', flush=True)
    except Exception as ex:
        print(f' [Skipped: {ex}]', flush=True)

if pb_all:
    de_res = pd.concat(pb_all, ignore_index=True)
    out_csv = os.path.join(DRIVE_OUT, 'pseudobulk_de_author38_sex_adjusted.csv')
    de_res.to_csv(out_csv, index=False)
    print(f'\n[SUCCESS] Pseudobulk DE complete! Output saved to Drive: {out_csv}', flush=True)


In [ ]:
# [5] Single-Cell Disease Relevance Scoring (scDRS) with Live Progress
import subprocess, sys

magma_path = os.path.join(INPUT_DIR, 'gene_level_cross_trait_all.tsv.gz')
if not os.path.exists(magma_path):
    raise FileNotFoundError(f'Missing {magma_path}. Please upload gene_level_cross_trait_all.tsv.gz to Drive.')

print('[Step 5A] Extracting top 1000 PGC MDD MAGMA Z-score genes...', flush=True)
magma = pd.read_csv(magma_path, sep='\t')
top_magma = (
    magma.dropna(subset=['SYMBOL', 'ZSTAT_MDD'])
    .sort_values('ZSTAT_MDD', ascending=False)
    .drop_duplicates('SYMBOL')
    .head(1000)
)
gs_df = pd.DataFrame({
    'TRAIT': ['MDD_PGC2025'],
    'GENESET': [','.join([f"{g}:{z:.3f}" for g, z in zip(top_magma['SYMBOL'], top_magma['ZSTAT_MDD'])])]
})
gs_path = os.path.join(DRIVE_OUT, 'mdd_magma_top1000.gs')
gs_df.to_csv(gs_path, sep='\t', index=False)
print(f'  -> Generated scDRS gene set with top {len(top_magma)} genes. Top 5: {top_magma["SYMBOL"].head(5).tolist()}', flush=True)

print('\n[Step 5B] Running scDRS compute-score on single nuclei (Live Streaming)...', flush=True)
local_h5ad = os.path.join(WORK_DIR, 'GSE213982_author41_clean.h5ad')
scdrs_cmd = [
    'scdrs', 'compute-score',
    '--h5ad-file', local_h5ad,
    '--h5ad-species', 'human',
    '--gs-file', gs_path,
    '--gs-species', 'human',
    '--cov-file', 'None',
    '--flag-filter-data', 'False',
    '--flag-raw-count', 'False',
    '--n-ctrl', '1000',
    '--flag-return-ctrl-raw-score', 'False',
    '--out-folder', DRIVE_OUT,
]

p = subprocess.Popen(scdrs_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
p.wait()
if p.returncode != 0:
    raise subprocess.CalledProcessError(p.returncode, scdrs_cmd)

print('\n[SUCCESS] scDRS scoring finished! Cell-level disease scores saved to Drive.', flush=True)


In [ ]:
# [6] Publication-Grade Single-Cell Visualization Suite (UMAP, Causal Genes, DEGs, scDRS)
import os, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
import scanpy as sc

# Ensure figures output directory exists in Google Drive
FIG_DIR = os.path.join(DRIVE_OUT, 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.linewidth'] = 0.8

print('[Step 6A] Generating 4-Panel Single-Nucleus UMAP Landscape (300 DPI)...', flush=True)
fig, axes = plt.subplots(2, 2, figsize=(14, 12), dpi=300)

# Panel A: UMAP by Major Cell Class
sc.pl.umap(adata, color='author_major', ax=axes[0, 0], show=False, frameon=False, title='A. Major Cortical Cell Classes', legend_loc='on data', legend_fontsize=8)

# Panel B: UMAP by Diagnosis (MDD vs Control)
sc.pl.umap(adata, color='diagnosis', palette={'Control': '#1f77b4', 'MDD': '#d62728'}, ax=axes[0, 1], show=False, frameon=False, title='B. Diagnosis (71 Donors: 35 Control vs 37 MDD)', legend_loc='right margin')

# Panel C: Aim 2 Lead Gene SLC12A5 (KCC2, chr20) Expression
if 'SLC12A5' in adata.var_names:
    sc.pl.umap(adata, color='SLC12A5', color_map='viridis', ax=axes[1, 0], show=False, frameon=False, title='C. SLC12A5 (KCC2, Shared Causal Variant with ANX)', vmin=0)
else:
    axes[1, 0].text(0.5, 0.5, 'SLC12A5 not found', ha='center', va='center')

# Panel D: Aim 2 Lead Gene DCC (chr18) Expression
if 'DCC' in adata.var_names:
    sc.pl.umap(adata, color='DCC', color_map='magma', ax=axes[1, 1], show=False, frameon=False, title='D. DCC (Netrin-1 Receptor, Distinct Causal Variants)', vmin=0)
else:
    axes[1, 1].text(0.5, 0.5, 'DCC not found', ha='center', va='center')

plt.tight_layout()
fig1_png = os.path.join(FIG_DIR, 'Figure_Aim1_DLPFC_snRNA_UMAP_Landscape.png')
fig1_pdf = os.path.join(FIG_DIR, 'Figure_Aim1_DLPFC_snRNA_UMAP_Landscape.pdf')
plt.savefig(fig1_png, dpi=300, bbox_inches='tight')
plt.savefig(fig1_pdf, bbox_inches='tight')
plt.show()
print(f'  -> Successfully saved Figure 1 to Drive: {fig1_png}', flush=True)

print('\n[Step 6B] Generating SLC12A5 & DCC Subtype Profiling and DE Summary...', flush=True)
fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(15, 6), dpi=300)

# Subtype-level ranking of SLC12A5 (KCC2)
if 'SLC12A5' in adata.var_names:
    sub_kcc2 = sc.get.obs_df(adata, keys=['SLC12A5', 'author_subtype']).groupby('author_subtype', observed=False).mean().sort_values('SLC12A5', ascending=True)
    top_kcc2 = sub_kcc2.tail(15)
    colors = ['#1b9e77' if 'InN' in s else '#7570b3' if 'ExN' in s else '#666666' for s in top_kcc2.index]
    ax2a.barh(np.arange(len(top_kcc2)), top_kcc2['SLC12A5'], color=colors, alpha=0.85, edgecolor='black', linewidth=0.5)
    ax2a.set_yticks(np.arange(len(top_kcc2)))
    ax2a.set_yticklabels(top_kcc2.index, fontsize=8)
    ax2a.set_xlabel('Mean Normalized Log1p Expression', fontweight='bold')
    ax2a.set_title('A. Top 15 Subtypes with Highest SLC12A5 (KCC2) Expression\n(Green: GABAergic Interneurons, Purple: Excitatory)', loc='left', fontweight='bold', fontsize=10)
    ax2a.spines['top'].set_visible(False)
    ax2a.spines['right'].set_visible(False)

# Pseudobulk PyDESeq2 Significant DEGs per Subtype
de_csv = os.path.join(DRIVE_OUT, 'pseudobulk_de_author38_sex_adjusted.csv')
if os.path.exists(de_csv):
    de_df = pd.read_csv(de_csv)
    if 'padj' in de_df.columns:
        sig_counts = de_df[de_df['padj'] < 0.05].groupby('cell_type').size().sort_values(ascending=True)
        top_sig = sig_counts.tail(15)
        ax2b.barh(np.arange(len(top_sig)), top_sig.values, color='#d95f02', alpha=0.85, edgecolor='black', linewidth=0.5)
        ax2b.set_yticks(np.arange(len(top_sig)))
        ax2b.set_yticklabels(top_sig.index, fontsize=8)
        ax2b.set_xlabel('Number of Significant DEGs (FDR < 0.05, MDD vs Control)', fontweight='bold')
        ax2b.set_title('B. Top Cortical Subtypes by Pseudobulk DE Perturbation', loc='left', fontweight='bold', fontsize=10)
        ax2b.spines['top'].set_visible(False)
        ax2b.spines['right'].set_visible(False)
        for i, v in enumerate(top_sig.values):
            ax2b.text(v + max(top_sig.values)*0.01, i, str(v), va='center', fontsize=7.5, fontweight='bold')
else:
    ax2b.text(0.5, 0.5, 'Pseudobulk DE results will appear after Cell [4] completes', ha='center', va='center')

plt.tight_layout()
fig2_png = os.path.join(FIG_DIR, 'Figure_Aim1_SLC12A5_DCC_CellType_Profiling.png')
fig2_pdf = os.path.join(FIG_DIR, 'Figure_Aim1_SLC12A5_DCC_CellType_Profiling.pdf')
plt.savefig(fig2_png, dpi=300, bbox_inches='tight')
plt.savefig(fig2_pdf, bbox_inches='tight')
plt.show()
print(f'  -> Successfully saved Figure 2 to Drive: {fig2_png}', flush=True)

# Step 6C: scDRS Disease Score Visualization (if Cell 5 finished)
scdrs_score_f = os.path.join(DRIVE_OUT, 'MDD_PGC2025.score.gz')
if os.path.exists(scdrs_score_f):
    print('\n[Step 6C] scDRS MDD disease scores detected! Plotting disease score distribution...', flush=True)
    df_scdrs = pd.read_csv(scdrs_score_f, sep='\t', index_col=0)
    common_cells = adata.obs_names.intersection(df_scdrs.index)
    adata.obs.loc[common_cells, 'scdrs_mdd_norm_score'] = df_scdrs.loc[common_cells, 'norm_score']
    
    fig3, (ax3a, ax3b) = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300)
    sc.pl.umap(adata, color='scdrs_mdd_norm_score', color_map='plasma', ax=ax3a, show=False, frameon=False, title='A. Single-Nucleus PGC MDD scDRS Disease Score', vmin=0)
    
    sub_scores = adata.obs.groupby('author_subtype', observed=False)['scdrs_mdd_norm_score'].mean().sort_values(ascending=True).dropna()
    top_sub_scores = sub_scores.tail(15)
    ax3b.barh(np.arange(len(top_sub_scores)), top_sub_scores.values, color='#7570b3', alpha=0.85, edgecolor='black', linewidth=0.5)
    ax3b.set_yticks(np.arange(len(top_sub_scores)))
    ax3b.set_yticklabels(top_sub_scores.index, fontsize=8)
    ax3b.set_xlabel('Mean scDRS Disease Score (MDD Risk Enrichment)', fontweight='bold')
    ax3b.set_title('B. Top 15 Subtypes Enriched for MDD Polygenic Risk', loc='left', fontweight='bold', fontsize=10)
    ax3b.spines['top'].set_visible(False)
    ax3b.spines['right'].set_visible(False)
    
    plt.tight_layout()
    fig3_png = os.path.join(FIG_DIR, 'Figure_Aim1_scDRS_CellType_Disease_Enrichment.png')
    fig3_pdf = os.path.join(FIG_DIR, 'Figure_Aim1_scDRS_CellType_Disease_Enrichment.pdf')
    plt.savefig(fig3_png, dpi=300, bbox_inches='tight')
    plt.savefig(fig3_pdf, bbox_inches='tight')
    plt.show()
    print(f'  -> Successfully saved Figure 3 to Drive: {fig3_png}', flush=True)
else:
    print('\n[Notice] Run Cell [5] anytime to compute scDRS disease scores and generate Figure 3.', flush=True)

print('\n[SUCCESS] Aim 1 publication figure generation complete! All figures saved in Drive.')